# Imports and Load

In [1]:
import os, json, joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

X_train = pd.read_parquet("../data/processed/X_train.parquet")
y_train = pd.read_parquet("../data/processed/y_train.parquet")["isFraud"]
X_test = pd.read_parquet("../data/processed/X_test.parquet")
y_test = pd.read_parquet("../data/processed/y_test.parquet")["isFraud"]

print(X_train.shape, X_test.shape)
print("train fraud rate:", y_train.mean(), "| test fraud rate:", y_test.mean())

(2217905, 12) (552504, 12)
train fraud rate: 0.0017832143396583713 | test fraud rate: 0.007706731534975312


# Evaluation Functions

In [2]:
results = []

def evaluate(name, model, X_te, y_te, threshold=0.5):
    proba = model.predict_proba(X_te)[:, 1]
    pred = (proba >= threshold).astype(int)
    results.append({
        "model": name,
        "PR-AUC": average_precision_score(y_te, proba),
        "precision": precision_score(y_te, pred, zero_division=0),
        "recall": recall_score(y_te, pred),
        "f1": f1_score(y_te, pred),
    })

def show_results():
    return pd.DataFrame(results).round(4).sort_values("PR-AUC", ascending=False)

# Pre Transaction Features

In [3]:
def add_pre_features(X):
    X = X.copy()
    X["amountToBalance"] = X["amount"] / (X["oldbalanceOrg"] + 1)
    X["drainsAccount"] = (X["amount"] >= X["oldbalanceOrg"]).astype(int)
    return X

pre_cols = ["amount", "oldbalanceOrg", "oldbalanceDest", "type_TRANSFER",
            "hourOfDay", "destWasEmpty", "amountToBalance", "drainsAccount"]

Xtr_pre = add_pre_features(X_train)[pre_cols]
Xte_pre = add_pre_features(X_test)[pre_cols]

# XGBoost Settings

In [4]:
params = dict(n_estimators=300, max_depth=6, learning_rate=0.1,
              tree_method="hist", eval_metric="aucpr", n_jobs=-1, random_state=42)
spw = (y_train == 0).sum() / (y_train == 1).sum()

In [ ]:
#Reference model: XGB with all features (leaky)
xgb_full = XGBClassifier(**params).fit(X_train, y_train)
evaluate("REF: XGB all features (leaky)", xgb_full, X_test, y_test)

pd.Series(xgb_full.feature_importances_, index=X_train.columns).sort_values(ascending=False)

destWasEmpty        0.392979
origEmptied         0.292815
type_TRANSFER       0.244252
newbalanceOrig      0.045947
newbalanceDest      0.010581
oldbalanceOrg       0.006189
amount              0.003788
errorBalanceOrig    0.001822
oldbalanceDest      0.001171
errorBalanceDest    0.000372
hourOfDay           0.000050
amountLog           0.000034
dtype: float32

In [ ]:
# Logistic Regression with pre transaction features
lr_pre = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
lr_pre.fit(Xtr_pre, y_train)
evaluate("LogReg pre-transaction", lr_pre, Xte_pre, y_test)

In [ ]:
#XGBoost , NO Resampling
xgb_plain = XGBClassifier(**params).fit(Xtr_pre, y_train)
evaluate("XGB pre plain", xgb_plain, Xte_pre, y_test)

In [ ]:
#XGBoost with class weight
xgb_cw = XGBClassifier(**params, scale_pos_weight=spw).fit(Xtr_pre, y_train)
evaluate("XGB pre class weight", xgb_cw, Xte_pre, y_test)

In [ ]:
#XGBoost with SMOTE
X_res, y_res = SMOTE(sampling_strategy=0.1, random_state=42).fit_resample(Xtr_pre, y_train)
print(y_res.value_counts())

xgb_sm = XGBClassifier(**params).fit(X_res, y_res)
evaluate("XGB pre SMOTE", xgb_sm, Xte_pre, y_test)

isFraud
0    2213950
1     221395
Name: count, dtype: int64


In [11]:
show_results()

,model,PR-AUC,precision,recall,f1
0,REF: XGB all features (leaky),1.0000,1.0000,0.9974,0.9987
4,XGB pre SMOTE,0.9998,0.9986,0.9993,0.9989
2,XGB pre plain,0.9892,0.9834,0.8643,0.9200
3,XGB pre class weight,0.9873,0.8693,0.9763,0.9197
1,LogReg pre-transaction,0.7486,0.9939,0.3840,0.5540


In [ ]:
final_model = xgb_sm #XGB with SMOTE is the best model based on PR-AUC

p = final_model.predict_proba(Xte_pre)[:, 1]
rows = []
for t in [0.1, 0.3, 0.5, 0.7, 0.9, 0.95]:
    pred = (p >= t).astype(int)
    rows.append({"threshold": t,
                 "precision": precision_score(y_test, pred, zero_division=0),
                 "recall": recall_score(y_test, pred),
                 "flagged": int(pred.sum())})
pd.DataFrame(rows).round(4)

,threshold,precision,recall,flagged
0,0.10,0.9863,0.9998,4316
1,0.30,0.9953,0.9998,4277
2,0.50,0.9986,0.9993,4261
3,0.70,0.9988,0.9988,4258
4,0.90,1.0000,0.9981,4250
5,0.95,1.0000,0.9977,4248


In [ ]:
REVIEW_T, BLOCK_T = 0.1, 0.9      # Chosen threshold from above table

decision = pd.cut(p, [-1, REVIEW_T, BLOCK_T, 2], labels=["approve", "review", "block"])
print(pd.crosstab(decision, y_test))

isFraud       0     1
row_0                
approve  548187     1
review       59     7
block         0  4250


In [19]:
os.makedirs("../models", exist_ok=True)
joblib.dump(final_model, "../models/fraud_xgb.joblib")
with open("../models/config.json", "w") as f:
    json.dump({"feature_columns": pre_cols,
               "review_threshold": REVIEW_T,
               "block_threshold": BLOCK_T}, f, indent=2)